# NB-09-01 — Image Processing Workbench

**Module M09 · Computer Vision** · [Lesson L09-02](../lessons/L09-02-image-processing-and-feature-extraction.md)

This notebook implements the classical pipeline from scratch on generated images: inspect the array,
blur it, find edges, threshold with Otsu, clean with morphology, and describe patches with HOG. Every
number is produced here — nothing is quoted from the lesson.

Run the cells in order. Each section ends with an exercise whose answer you should predict before running
the next cell.

In [1]:
import sys
sys.path.insert(0, "../code")          # the module's reference code lives one directory up

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import vision_core as v

print("vision_core self-test:", v.run_tests())

vision_core self-test: True


## 1. An image is an array

Shape is `(rows, columns, channels)`; coordinates are `(y, x)`. Getting that wrong is the most common
vision bug, so check it explicitly before doing anything else.

In [2]:
image, mask = v.make_road_scene(seed=3)
print("shape:", image.shape, "dtype:", image.dtype, "range:", image.min(), image.max())
print("one pixel (row 10, col 12):", image[10, 12])

gray = v.to_gray(image)
print("luma shape:", gray.shape, "mean:", round(float(gray.mean()), 2), "std:", round(float(gray.std()), 2))

naive = image.mean(axis=2)
print("max |luma - plain mean|:", round(float(np.abs(gray - naive).max()), 2))

shape: (64, 64, 3) dtype: uint8 range: 42 228
one pixel (row 10, col 12): [119 167 224]
luma shape: (64, 64) mean: 120.5 std: 32.14
max |luma - plain mean|: 16.15


The weighted luma and the plain channel mean differ: green counts almost twice as much as blue.
For a mostly green leaf that is a visible brightness difference, not a rounding detail.

## 2. Convolution and padding

Output size is $\lfloor (n + 2p - k)/s \rfloor + 1$. Check `same` and `valid` against the formula.

In [3]:
kernel = np.ones((3, 3)) / 9
same = v.convolve2d(gray, kernel)
valid = v.convolve2d(gray, kernel, "valid")
print("same:", same.shape, "| valid:", valid.shape, "| expected valid:", (gray.shape[0] - 2, gray.shape[1] - 2))

identity = np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=float)
print("identity preserves the image:", np.allclose(v.convolve2d(gray, identity), gray))

same: (64, 64) | valid: (62, 62) | expected valid: (62, 62)
identity preserves the image: True


**Exercise.** Predict the shape for a 5×5 kernel with `same` and `valid` on a 64×64 image before running
the next cell.

In [4]:
print(v.convolve2d(gray, np.ones((5, 5)) / 25).shape, v.convolve2d(gray, np.ones((5, 5)) / 25, "valid").shape)

(64, 64) (60, 60)


## 3. Blurring and sharpening change variance

A blur removes high frequency, so variance falls. A sharpen amplifies it. The numbers make the claim
checkable.

In [5]:
blurred = v.convolve2d(gray, v.gaussian_kernel(5, 1.0))
sharp = v.convolve2d(blurred, np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=float))
print("original variance:", round(float(gray.var()), 1))
print("blurred variance: ", round(float(blurred.var()), 1))
print("sharpened variance:", round(float(sharp.var()), 1))

original variance: 1033.1
blurred variance:  957.9
sharpened variance: 1135.7


## 4. Edges, Otsu and morphology

Sobel magnitude, then Otsu's threshold, then an opening to remove specks. Count what survives.

In [6]:
edges = v.sobel_magnitude(blurred)
threshold = v.otsu_threshold(edges)
raw = edges > threshold
opened = v.dilate(v.erode(raw, 3), 3)
print("otsu threshold:", threshold)
print("pixels above threshold:", int(raw.sum()), "-> after opening:", int(opened.sum()))

otsu threshold: 101.61585557178336
pixels above threshold: 255 -> after opening: 0


## 5. A HOG descriptor

Length is $(H/c)(W/c)b$: for 64×64, cells of 8 and 9 bins that is $8 \cdot 8 \cdot 9 = 576$.

In [7]:
descriptor = v.hog_descriptor(gray, cell=8, bins=9)
print("length:", descriptor.size, "expected:", (gray.shape[0] // 8) * (gray.shape[1] // 8) * 9)
print("first five values:", np.round(descriptor[:5], 3))

length: 576 expected: 576
first five values: [0.345 0.014 0.174 0.01  0.802]


## 6. The baseline and its blind spot

A colour rule is a legitimate baseline. It is also a warning: pixel accuracy stays high while an entire
class is never predicted.

In [8]:
baseline = np.where(image[:, :, 2] > 150, 1, 0).astype(np.uint8)
print("pixel accuracy:", round(float((baseline == mask).mean()), 4))
print("mean IoU:", round(v.mean_iou(baseline, mask), 3))
print("obstacle Dice:", round(v.dice(baseline, mask, 2), 3))
print("obstacle share of pixels:", round(float((mask == 2).mean()), 4))

pixel accuracy: 0.9797
mean IoU: 0.65
obstacle Dice: 0.0
obstacle share of pixels: 0.0137


## 7. Exercise: improve the rule

Obstacles are brown (red clearly above blue). Add that rule, then measure again. Predict first: which of
the three numbers will change most?

Then answer: why can a rule look reasonable and still score 0.0 on the class that matters?

In [9]:
better = np.zeros_like(mask)
better[image[:, :, 2] > 150] = 1
brown = (image[:, :, 0].astype(int) - image[:, :, 2].astype(int)) > 40
better[brown] = 2
print("mIoU:", round(v.mean_iou(better, mask), 3))
for label in (0, 1, 2):
    print(label, "Dice:", round(v.dice(better, mask, label), 3))

mIoU: 0.991
0 Dice: 0.995
1 Dice: 0.991
2 Dice: 1.0


## Self-assessment

1. Why does `same` padding with an even kernel size shift the output?
2. What does opening remove that closing does not?
3. State the two numbers that exposed the baseline's failure, and why accuracy did not.

Write your answers before reading the next notebook; the lab asks for the same reasoning in prose.